# Lecture: Building a Reproducible Analysis Workflow

Use three Pokémon questions to practice a reproducible analysis. We will predict results, introduce Pandas operations as they are needed, and connect each conclusion to checked evidence.


## Learning goals

- Organize an analysis using the seven-step workflow.
- Identify observations and relevant features.
- Group records and compare means with group counts.
- Use vectorized comparisons and numeric indicators to calculate percentages.
- Distinguish absent secondary types from unknown data using documentation.
- Check results and state conclusions and limitations.


## The reproducible analysis workflow

We will use seven steps:

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Operation:** What should the code select, calculate, or compare?
4. **Check:** Does the result make sense in context? Do a few original records support the meaning of the calculation?
5. **Evidence:** Which result directly answers the question?
6. **Conclusion:** What claim is supported?
7. **Limitation:** What should we avoid concluding?


Code that runs without an error can still answer the wrong question. A reproducible analysis preserves the code and records how each result supports the conclusion.


# Part 1: Pokémon base stats


## Background

Use Alberto Barradas's [Pokémon with stats dataset on Kaggle](https://www.kaggle.com/datasets/abcsds/pokemon). A fixed copy is supplied as `data/pokemon_stats.csv`, so everyone analyzes the same records without needing a Kaggle account.

The file has 800 records representing Pokémon and alternate forms from Generations 1–6, with 721 distinct Pokédex numbers. One row is a listed Pokémon/form, not a battle or an individual player's Pokémon. Alternate forms can share a Pokédex number; do not assume repeated numbers are accidental duplicates. This historical file does not cover every current generation.

Base stats are game attributes. `Total` is the sum of six base stats, not a win rate or a guarantee of battle success. No Pokémon expertise is required: use the dictionary and definitions below.


### Data dictionary

| Feature | Description |
| --- | --- |
| `#` | Pokédex number; an identifier that can repeat for alternate forms |
| `Name` | Name of the listed Pokémon/form |
| `Type 1` | Primary type in this file |
| `Type 2` | Secondary type; blank for records with no secondary type |
| `Total` | Sum of HP, Attack, Defense, Sp. Atk, Sp. Def, and Speed |
| `HP` | Base hit points |
| `Attack` | Base physical attack stat |
| `Defense` | Base physical defense stat |
| `Sp. Atk` | Base special attack stat |
| `Sp. Def` | Base special defense stat |
| `Speed` | Base speed stat |
| `Generation` | Generation of introduction recorded in this file |
| `Legendary` | True/False classification supplied by the dataset author |

Use the dataset's classifications as recorded. Primary type alone does not capture all type combinations or battle matchups.


## Scope for all three investigations

Use all 800 records, including Legendary Pokémon. Count alternate forms as individual Pokémon for this lecture: each row receives equal weight, and repeated Pokédex numbers are retained. Use the generation labels recorded in the file. These questions concern this historical dataset, not every Pokémon released since it was assembled.


## Investigation 1: Generation and average HP

### Step 1: Question

**Which generation of Pokémon (including alternative forms as individual Pokémon) has the highest average HP?**

Use the arithmetic mean as the average.


### Prediction — before calculating


**Which generation do you predict will have the highest average HP, and why? Record your prediction before running the analysis.**


### Step 2: Data


Import Pandas as `pd`, read `data/pokemon_stats.csv` into `pokemon_df`, and display its first and last five rows, shape, `.info()`, and missing-value counts.


In [ ]:
import pandas as pd


In [ ]:
pokemon_df = pd.read_csv("data/pokemon_stats.csv")


In [ ]:
pokemon_df.head()


In [ ]:
pokemon_df.tail()


In [ ]:
pokemon_df.shape


In [ ]:
pokemon_df.info()


In [ ]:
pokemon_df.isnull().sum()


**What does one row represent, which features answer this question, and are those features complete?**


### Step 3: Operation


`groupby()` partitions the rows by a category. `.agg(["count", "mean"])` calculates two summaries of HP for each group. Counts show how many records contribute to each mean; `.sort_values()` orders the summary without changing those means.


Group `pokemon_df` by `Generation`. Summarize `HP` using `.agg(["count", "mean"])`, name the result `generation_hp_df`, and rename its columns `pokemon_count` and `mean_hp`. Sort from greatest to smallest mean HP.


In [ ]:
generation_hp_df = pokemon_df.groupby("Generation")["HP"].agg(["count", "mean"])


In [ ]:
generation_hp_df.columns = ["pokemon_count", "mean_hp"]


In [ ]:
generation_hp_df = generation_hp_df.sort_values("mean_hp", ascending=False)


### Step 4: Check


Before trusting the ranking, ask whether its values make sense as **average HP**, rather than totals or counts. Inspect a few Generation 4 records and compare their HP values with that generation’s mean. You do not need to recognize the Pokémon to read the values.


In [ ]:
pokemon_df.loc[pokemon_df["Name"].isin(["Turtwig", "Chimchar", "Piplup", "Garchomp", "Lucario", "Arceus"]), ["Name", "Generation", "HP"]]


In [ ]:
generation_hp_df.loc[4, "mean_hp"]


**Does an average around this size seem plausible alongside these individual HP values? What would make you suspicious—for example, a negative average or one in the thousands? Does a surprising result necessarily mean the code is wrong?**


### Step 5: Evidence


Display the full sorted summary, rounded to two decimal places. Round for display only.


In [ ]:
generation_hp_df.round(2)


**Which generation has the highest mean HP? Report its mean and number of records.**


### Step 6: Conclusion


**Write a sentence answering the question, then explain whether the evidence supports your prediction.**


### Step 7: Limitation


**Why does this result not mean that every Generation 4 Pokémon has higher HP than every Pokémon in other generations?**


## Investigation 2: Secondary types

### Step 1: Question

**What percent of Pokémon (including alternative forms as individual Pokémon) have a secondary type?**


**Before calculating, do you predict that fewer than half, about half, or more than half have a secondary type? Explain briefly.**


### Step 2: Data

Reuse all rows in `pokemon_df` and the `Type 2` feature. According to the dictionary, a blank secondary type means no secondary type in this file. Retain those rows in the denominator; they are part of the question.


### Step 3: Operation


`.notna()` produces one Boolean per row. `.astype(int)` converts True to 1 and False to 0. The mean of a 0/1 indicator is the fraction of 1s, so multiplying by 100 gives a percentage. Assigning a Series to a new column aligns it with the DataFrame rows.


Create `has_secondary_type` directly in `pokemon_df` using `.notna().astype(int)` on `Type 2`. Calculate `secondary_type_count` by summing the indicator and `secondary_type_percent` by taking its mean and multiplying by 100.


In [ ]:
pokemon_df["has_secondary_type"] = pokemon_df["Type 2"].notna().astype(int)


In [ ]:
secondary_type_count = pokemon_df["has_secondary_type"].sum()


In [ ]:
secondary_type_percent = pokemon_df["has_secondary_type"].mean() * 100


### Step 4: Check


Read a few original type labels beside the new indicator. For each row, decide what the indicator should be before comparing it with the calculated value. Is the code answering “has a secondary type,” or did we accidentally reverse that meaning?


In [ ]:
pokemon_df.loc[pokemon_df["Name"].isin(["Bulbasaur", "Charmander", "Squirtle", "Charizard"]), ["Name", "Type 1", "Type 2", "has_secondary_type"]]


**Which rows should have 1 and which should have 0? Would a reported 100% make sense when the file includes Pokémon with no secondary type? What is the difference between a proportion near 0.52 and a percentage near 52?**


### Step 5: Evidence


Display the numerator, denominator, and percentage rounded to two decimal places.


In [ ]:
print("With secondary type:", secondary_type_count)


In [ ]:
print("All records:", len(pokemon_df))


In [ ]:
print("Percent:", round(secondary_type_percent, 2))


### Step 6: Conclusion


**Answer the research question in one sentence using both the percentage and counts.**


### Step 7: Limitation


**Why should we consult the dictionary before interpreting blank values, and what population does this percentage describe?**


## Investigation 3: Attack compared with Defense

### Step 1: Question

**What percent of Pokémon (including alternative forms as individual Pokémon) have an Attack that is higher than, lower than, or equal to their Defense?**

Use the physical `Attack` and `Defense` base stats, not the special stats. In the main analysis, **equal means exactly equal**.


**Before calculating, which group do you predict will be largest? Do you expect many exact ties?**


### Step 2: Data

Reuse `Attack` and `Defense` from the initial inspection. Both are numeric base-stat measurements. We will compare the two values within each row.


### Step 3: Operation


Comparing two Series performs an element-by-element comparison for aligned rows. `.loc[condition, "column"]` assigns labels only where the condition is True. With complete numeric values, greater than, less than, and exact equality divide the rows into three non-overlapping groups.


Create three Boolean conditions: `attack_higher`, `attack_lower`, and `attack_equal`. Use `>`, `<`, and `==`. Create `attack_defense_group` in `pokemon_df`, then use `.loc` to assign readable labels for the three conditions.


In [ ]:
attack_higher = pokemon_df["Attack"] > pokemon_df["Defense"]


In [ ]:
attack_lower = pokemon_df["Attack"] < pokemon_df["Defense"]


In [ ]:
attack_equal = pokemon_df["Attack"] == pokemon_df["Defense"]


In [ ]:
pokemon_df["attack_defense_group"] = pd.Series(index=pokemon_df.index, dtype="object")


In [ ]:
pokemon_df.loc[attack_higher, "attack_defense_group"] = "Higher"


In [ ]:
pokemon_df.loc[attack_lower, "attack_defense_group"] = "Lower"


In [ ]:
pokemon_df.loc[attack_equal, "attack_defense_group"] = "Equal"


Create `attack_defense_summary_df` with counts from `.value_counts()` and percentages calculated as each count divided by the total number of rows times 100.


In [ ]:
attack_defense_summary_df = pokemon_df["attack_defense_group"].value_counts().to_frame("pokemon_count")


In [ ]:
attack_defense_summary_df["percent"] = attack_defense_summary_df["pokemon_count"] / len(pokemon_df) * 100


### Step 4: Check


Look at a few records from each assigned group. Compare Attack and Defense yourself before reading the label. This checks whether “Higher” means Attack is higher than Defense—not the other way around—and whether exact ties have their own group.


In [ ]:
pokemon_df.loc[pokemon_df["attack_defense_group"] == "Higher", ["Name", "Attack", "Defense", "attack_defense_group"]].head(3)


In [ ]:
pokemon_df.loc[pokemon_df["attack_defense_group"] == "Lower", ["Name", "Attack", "Defense", "attack_defense_group"]].head(3)


In [ ]:
pokemon_df.loc[pokemon_df["attack_defense_group"] == "Equal", ["Name", "Attack", "Defense", "attack_defense_group"]].head(3)


**Do the labels match the visible values? Where should a record with Attack 50 and Defense 50 go? What error might put that record in Higher or Lower?**


### Step 5: Evidence


Display the counts and percentages. Round the displayed percentages to two decimal places.


In [ ]:
attack_defense_summary_df.round(2)


### Step 6: Conclusion


**Write a sentence reporting all three percentages and their counts.**


### Step 7: Limitation


**Why does higher Attack than Defense not prove that a Pokémon wins more battles?**


## Summary

We answered three questions using all 800 listed Pokémon/forms. Grouped means compared generations, an indicator measured the share with a secondary type, and vectorized comparisons classified Attack relative to Defense. We inspected the data once, then used targeted sanity checks to examine plausible HP values, indicator meanings, percentage scale, and comparison labels. A sanity check can reveal a mistake without proving every result correct. The lab applies the same workflow to a dataset you choose.
